In [1]:
import pandas as pd
import lxml
import requests
import yfinance as yf
from io import StringIO

In [2]:
url= "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers= {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [3]:
response=requests.get(url, headers=headers)

sp500=pd.read_html(StringIO(response.text))[0]
sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [4]:
sp500['Symbol']=sp500['Symbol'].str.replace(".", "-", regex=False)

In [5]:
# list of ticker symbols for download (Needs to be done for S&P 500)
tickerStrings = sp500['Symbol'].tolist()

In [6]:
df = yf.download(tickerStrings, group_by='Ticker', period='1d')

[*********************100%***********************]  503 of 503 completed


In [7]:
df

Ticker            CIEN                                          GEN  \
Price             Open   High        Low       Close   Volume  Open   
Date                                                                  
2026-06-22  437.829987  461.5  430.51001  460.329987  3857000  23.5   

Ticker                                             ...         CPT  \
Price            High        Low  Close    Volume  ...        Open   
Date                                               ...               
2026-06-22  24.120001  22.459999  23.01  11780100  ...  109.980003   

Ticker                                                     LMT         \
Price             High         Low       Close   Volume   Open   High   
Date                                                                    
2026-06-22  109.980003  107.970001  108.980003  1182300  508.5  510.0   

Ticker                                       
Price              Low       Close   Volume  
Date                                         
2026-06-22  490.049988  493.600006  1703200  

[1 rows x 2515 columns]

In [8]:
df = df.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
df

Price,Date,Ticker,Open,High,Low,Close,Volume
0,2026-06-22,CIEN,437.829987,461.500000,430.510010,460.329987,3857000
1,2026-06-22,GEN,23.500000,24.120001,22.459999,23.010000,11780100
2,2026-06-22,HON,229.779999,231.399994,227.199997,228.110001,4028900
3,2026-06-22,KO,79.250000,80.059998,79.059998,79.529999,21199100
4,2026-06-22,YUM,150.500000,152.000000,149.509995,150.740005,2473800
...,...,...,...,...,...,...,...
498,2026-06-22,ARE,51.389999,51.389999,50.349998,50.730000,1175300
499,2026-06-22,ADM,75.449997,76.519997,74.519997,76.290001,3567700
500,2026-06-22,VMC,300.660004,307.029999,298.570007,304.390015,968900
501,2026-06-22,CPT,109.980003,109.980003,107.970001,108.980003,1182300


In [9]:
downloaded_tickers=set(df["Ticker"].unique())

In [10]:
missing = set(tickerStrings) - downloaded_tickers

missing

set()

In [11]:
len(downloaded_tickers)

503

In [12]:
row_counts = df.groupby("Ticker").size()
print(row_counts.sort_values())

Ticker
A       1
NWS     1
NVR     1
NVDA    1
NUE     1
       ..
ELV     1
EL      1
EIX     1
ETR     1
ZTS     1
Length: 503, dtype: int64


In [13]:
for ticker in ["FRT", "PHM", "SYY"]:
    count = (df["Ticker"] == ticker).sum()
    print(f"{ticker}: {count} rows")

FRT: 1 rows
PHM: 1 rows
SYY: 1 rows


In [14]:
df.columns.name = None
df

,Date,Ticker,Open,High,Low,Close,Volume
0,2026-06-22,CIEN,437.829987,461.500000,430.510010,460.329987,3857000
1,2026-06-22,GEN,23.500000,24.120001,22.459999,23.010000,11780100
2,2026-06-22,HON,229.779999,231.399994,227.199997,228.110001,4028900
3,2026-06-22,KO,79.250000,80.059998,79.059998,79.529999,21199100
4,2026-06-22,YUM,150.500000,152.000000,149.509995,150.740005,2473800
...,...,...,...,...,...,...,...
498,2026-06-22,ARE,51.389999,51.389999,50.349998,50.730000,1175300
499,2026-06-22,ADM,75.449997,76.519997,74.519997,76.290001,3567700
500,2026-06-22,VMC,300.660004,307.029999,298.570007,304.390015,968900
501,2026-06-22,CPT,109.980003,109.980003,107.970001,108.980003,1182300


In [28]:
def validate_download(df, tickerStrings, ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]):
    
    # Check 1: tickers entirely missing from df
    downloaded_tickers = set(df["Ticker"].unique())
    fully_missing = set(tickerStrings) - downloaded_tickers
    
    # Check 2: tickers present but with any null OHLCV values
    null_counts = (
        df.groupby("Ticker")[ohlcv_cols]
        .apply(lambda x: x.isna().sum().sum())
    )
    has_nulls = null_counts[null_counts > 0]
    
    # Check 3: tickers where ALL rows are null (completely empty shells)
    row_counts = df.groupby("Ticker").size()
    total_cells = row_counts * len(ohlcv_cols)
    fully_null = null_counts[null_counts == total_cells[null_counts.index]]
    partially_null = has_nulls[~has_nulls.index.isin(fully_null.index)]
    
    # Report
    print(f"Total tickers expected: {len(tickerStrings)}")
    print(f"Total tickers downloaded: {len(downloaded_tickers)}")
    
    if fully_missing:
        print(f"\nFully missing (not in df at all): {fully_missing}")
    else:
        print("\nFully missing: none")
    
    if len(fully_null) > 0:
        print(f"\nPresent but completely null (failed download): {fully_null.index.tolist()}")
    else:
        print("Completely null tickers: none")

    if len(partially_null) > 0:
        print(f"\nPartially null (recent IPO/spin-off, expected): \n{partially_null}")
    else:
        print("Partially null tickers: none")

    return fully_missing, fully_null, partially_null

In [29]:
fully_missing, fully_null, partially_null = validate_download(df, tickerStrings)

Total tickers expected: 503
Total tickers downloaded: 503

Fully missing: none
Completely null tickers: none
Partially null tickers: none


In [20]:
df.to_csv('daily_data_2306.csv')

In [22]:
null_counts_by_ticker = (
    df.groupby("Ticker")[ohlcv_cols]
    .apply(lambda x: x.isna().sum().sum())  # total nulls across all OHLCV cols, per ticker
    .sort_values(ascending=False)
)

print(null_counts_by_ticker.head(10))

Ticker
A       0
NI      0
NXPI    0
NWSA    0
NWS     0
NVR     0
NVDA    0
NUE     0
NTRS    0
NTAP    0
dtype: int64
